<a href="https://colab.research.google.com/github/0xFAB10/industrial-sentinel-lite/blob/main/Experimento_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.9 MB/s eta 0:00:00


In [3]:
import os
import platform
import torch
import ultralytics
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15
Ultralytics: 8.4.174
PyTorch: 2.11.0+cu130
CUDA disponível: True
GPU: Tesla T4


In [5]:
from pathlib import Path

PROJECT_DIR = Path("/content/industrial-sentinel-lite") if Path("/content").exists() else Path.cwd()
RUNS_DIR = PROJECT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = "construction-ppe.yaml"
MODEL_NAME = "yolo26n.pt"
EPOCHS = 10
IMAGE_SIZE = 640
BATCH_SIZE = 8

print("Diretório de trabalho:", PROJECT_DIR)
print("Dataset:", DATA_YAML)
print("Modelo inicial:", MODEL_NAME)

Diretório de trabalho: /content/industrial-sentinel-lite
Dataset: construction-ppe.yaml
Modelo inicial: yolo26n.pt


In [6]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(RUNS_DIR),
    name="construction_ppe_baseline",
    exist_ok=True,
    plots=True,
    patience=5,
    seed=42
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=construction-ppe.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=construction_ppe_baseline, nbs=64, nms=None, opset=None, optimize=Fals

In [7]:
BEST_WEIGHTS = RUNS_DIR / "construction_ppe_baseline" / "weights" / "best.pt"
LAST_WEIGHTS = RUNS_DIR / "construction_ppe_baseline" / "weights" / "last.pt"

print("Melhor peso existe:", BEST_WEIGHTS.exists(), BEST_WEIGHTS)
print("Último peso existe:", LAST_WEIGHTS.exists(), LAST_WEIGHTS)

if BEST_WEIGHTS.exists():
    trained_model = YOLO(str(BEST_WEIGHTS))
else:
    raise FileNotFoundError("best.pt não foi encontrado. Confira a saída da célula de treinamento.")

Melhor peso existe: True /content/industrial-sentinel-lite/runs/construction_ppe_baseline/weights/best.pt
Último peso existe: True /content/industrial-sentinel-lite/runs/construction_ppe_baseline/weights/last.pt


In [8]:
val_results = trained_model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMAGE_SIZE,
    plots=True,
    project=str(RUNS_DIR),
    name="construction_ppe_validation"
)

print(val_results)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,376,981 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2799.1±1237.8 MB/s, size: 163.6 KB)
val: Scanning /content/datasets/construction-ppe/labels/val.cache... 143 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 143/143 54.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.4it/s 3.7s
                   all        143       1172      0.757      0.493       0.53      0.261
                helmet        107        201       0.75      0.806      0.815      0.421
                gloves         68        136      0.718      0.735      0.768      0.341
                  vest        109        171      0.677      0.813      0.843      0.509
                 boots         64        151      0.687      0.748      0.789      0.445
               goggles       

In [9]:
test_results = trained_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMAGE_SIZE,
    plots=True,
    project=str(RUNS_DIR),
    name="construction_ppe_test"
)

print(test_results)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,376,981 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1594.8±659.8 MB/s, size: 55.6 KB)
val: Scanning /content/datasets/construction-ppe/labels/test... 141 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 141/141 1.8Kit/s 0.1s
val: New cache created: /content/datasets/construction-ppe/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.3it/s 3.9s
                   all        141       1251      0.582      0.492      0.501      0.254
                helmet        110        192      0.869      0.906      0.929      0.492
                gloves         75        163      0.737      0.703      0.743      0.355
                  vest        111        178      0.779      0.899       0.92      0.565
                 boots         75        211     

In [10]:
from IPython.display import display, Image
import glob

dataset_candidates = [
    Path.home() / "datasets" / "construction-ppe",
    Path("/content/datasets/construction-ppe"),
    Path.cwd() / "datasets" / "construction-ppe",
]

dataset_dir = next((p for p in dataset_candidates if p.exists()), None)

if dataset_dir is None:
    print("Não encontrei a pasta do dataset nos caminhos comuns.")
    print("Procure a pasta construction-ppe no ambiente e ajuste dataset_dir manualmente.")
else:
    test_images = sorted((dataset_dir / "images" / "test").glob("*"))
    test_images = [p for p in test_images if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    print("Dataset:", dataset_dir)
    print("Imagens de teste encontradas:", len(test_images))

    if test_images:
        sample_images = test_images[:5]
        prediction_results = trained_model.predict(
            source=[str(p) for p in sample_images],
            imgsz=IMAGE_SIZE,
            conf=0.25,
            save=True,
            project=str(RUNS_DIR),
            name="sample_predictions",
            exist_ok=True
        )
        print("Imagens anotadas salvas em:", RUNS_DIR / "sample_predictions")

Dataset: /content/datasets/construction-ppe
Imagens de teste encontradas: 141

0: 640x640 1 helmet, 3 glovess, 1 vest, 1 Person, 6.9ms
1: 640x640 1 helmet, 4 vests, 5 bootss, 3 Persons, 6.9ms
2: 640x640 1 helmet, 2 glovess, 1 vest, 2 bootss, 1 goggles, 1 Person, 6.9ms
3: 640x640 1 helmet, 3 glovess, 1 vest, 3 bootss, 1 goggles, 1 Person, 6.9ms
4: 640x640 1 helmet, 1 gloves, 1 vest, 1 boots, 1 Person, 6.9ms
Speed: 1.2ms preprocess, 6.9ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/industrial-sentinel-lite/runs/sample_predictions
Imagens anotadas salvas em: /content/industrial-sentinel-lite/runs/sample_predictions
